# Phase 3d: Layer 3, Blind Passes

**Project:** A Perception-Aware Framework for Evaluating Counterfactual Passing Options in Professional Football

**Author:** Raeesa Lorgat (2686242)

**Purpose of this notebook:** test whether passes to teammates outside the passer's vision cone (blind passes) go to teammates whose position could be recovered from short-term memory, the second half of RQ3.

Identifying the receiver of each pass means matching the pass's end location to a teammate's freeze-frame position, so the coordinate systems involved have to agree. The first section checks this. Freeze frames and StatsBomb events use a 120 by 80 yard pitch drawn from the acting team's point of view, while the socceraction (SPADL) actions used for VAEP use a 105 by 68 metre pitch with their own orientation.

In [8]:
import warnings
warnings.filterwarnings('ignore')

import glob
import time
from pathlib import Path

import numpy as np
import pandas as pd
from statsbombpy import sb
import statsbombpy.public as sbpublic

sbpublic.OPEN_DATA_PATHS = {
    k: v.replace(
        "https://raw.githubusercontent.com/statsbomb/open-data/master",
        "https://cdn.jsdelivr.net/gh/statsbomb/open-data@master",
    )
    for k, v in sbpublic.OPEN_DATA_PATHS.items()
}
pd.set_option('display.max_columns', None)

CACHE_DIR = Path("../data/statsbomb_cache")

def cached_events(match_id, retries=5, backoff=5):
    cache_file = CACHE_DIR / f"events_{match_id}.pkl"
    if cache_file.exists():
        return pd.read_pickle(cache_file)
    for attempt in range(retries):
        try:
            events = sb.events(match_id=match_id)
            events.to_pickle(cache_file)
            return events
        except Exception as e:
            if attempt == retries - 1:
                raise
            time.sleep(backoff * (attempt + 1))


def load_match_actions(mid):
    """The SPADL actions saved for this match in setup_euro2024.ipynb."""
    for folder in ['../data/matches', '../data/matches_extended']:
        p = Path(folder) / f'match_{mid}.pkl'
        if p.exists():
            return pd.read_pickle(p)['actions']
    raise FileNotFoundError(f"no saved actions for match {mid}")


all_match_ids = [f.split('lstm_eval_match_')[-1].replace('.pkl', '')
                 for f in sorted(glob.glob('../data/lstm_eval_all3/lstm_eval_match_*.pkl'))]
check_ids = all_match_ids[:5]
print(f"{len(all_match_ids)} matches, checking {check_ids}")

166 matches, checking ['3788741', '3788742', '3788743', '3788744', '3788745']


Check 1: how SPADL coordinates relate to StatsBomb coordinates

For every pass in five matches, the SPADL start location is compared with the StatsBomb event location after rescaling from 120 by 80 yards to 105 by 68 metres. Four versions are compared: x as it is or mirrored end to end, and y as it is or flipped top to bottom. Whichever version gives a median difference close to zero is the true relationship, and it may differ between the two teams in a match.

In [9]:
rows = []
for mid in check_ids:
    actions = load_match_actions(mid)
    ev = cached_events(mid)
    ev = ev[ev['type'] == 'Pass'][['id', 'team', 'location']]
    p = actions[actions['type_name'] == 'pass'].merge(ev, left_on='original_event_id', right_on='id')

    x_scaled = (p['location'].str[0] - 1) / 119 * 105
    y_scaled = (p['location'].str[1] - 1) / 79 * 68
    p = p.assign(
        x_as_is=(p['start_x'] - x_scaled).abs(),
        x_mirrored=(p['start_x'] - (105 - x_scaled)).abs(),
        y_as_is=(p['start_y'] - y_scaled).abs(),
        y_flipped=(p['start_y'] - (68 - y_scaled)).abs(),
    )
    summary = p.groupby('team')[['x_as_is', 'x_mirrored', 'y_as_is', 'y_flipped']].median()
    summary['passes'] = p.groupby('team').size()
    summary['match_id'] = mid
    rows.append(summary)

print("Median difference (m) between SPADL start and rescaled StatsBomb location:")
print(pd.concat(rows).round(2).to_string())

Median difference (m) between SPADL start and rescaled StatsBomb location:
             x_as_is  x_mirrored  y_as_is  y_flipped  passes match_id
team                                                                 
Italy          36.07        0.35     0.40      34.35     598  3788741
Turkey          0.52       40.11    29.94       0.37     346  3788741
Denmark         0.33       26.45    40.59       0.40     610  3788742
Finland        37.92        0.53     0.41      33.10     262  3788742
Belgium         0.44       27.37    35.54       0.45     701  3788743
Russia         35.10        0.46     0.37      36.74     347  3788743
Switzerland    30.49        0.42     0.39      30.79     504  3788744
Wales           0.47       41.39    33.49       0.41     280  3788744
Croatia        30.84        0.43     0.33      31.27     428  3788745
England         0.43       30.18    39.99       0.51     432  3788745


Check 2: does the receiver's hypothetical value match the real pass?

If candidate values were computed in the right coordinates, the hypothetical pass to the player who actually received the ball should be worth about the same as the real pass, since it is nearly the same action. The receiver is taken as the teammate whose freeze-frame position is closest to the StatsBomb pass end location, within 5 yards. The gap between the receiver's candidate value and the real value is then compared between the two teams in each match. A coordinate mismatch would show up as a gap that is consistently different from zero, and possibly much larger for one team than the other.

In [10]:
candidates_full = pd.read_pickle('../data/results/candidates_full_old_coords.pkl')   # the pre-fix table
candidates_full['match_id'] = candidates_full['match_id'].astype(str)

res = []
for mid in check_ids:
    ev = cached_events(mid)
    passes = ev[ev['type'] == 'Pass'][['id', 'team', 'pass_end_location']]
    c = candidates_full[candidates_full['match_id'] == mid].merge(
        passes, left_on='original_event_id', right_on='id')
    c['d_end'] = np.hypot(c['cand_x'] - c['pass_end_location'].str[0],
                          c['cand_y'] - c['pass_end_location'].str[1])
    rec = c.loc[c.groupby('original_event_id')['d_end'].idxmin()]
    rec = rec[rec['d_end'] < 5].copy()
    rec['value_gap'] = rec['candidate_value'] - rec['real_value']
    res.append(rec[['match_id', 'team', 'd_end', 'candidate_value', 'real_value', 'value_gap']])

rec_all = pd.concat(res)
print(f"{len(rec_all)} passes with a receiver found within 5 yards")
print(rec_all.groupby(['match_id', 'team']).agg(
    passes=('value_gap', 'size'),
    mean_real=('real_value', 'mean'),
    mean_candidate=('candidate_value', 'mean'),
    median_gap=('value_gap', 'median'),
    median_abs_gap=('value_gap', lambda s: s.abs().median()),
).round(4).to_string())

1294 passes with a receiver found within 5 yards
                      passes  mean_real  mean_candidate  median_gap  median_abs_gap
match_id team                                                                      
3788741  Italy           219    -0.0010          0.0085      0.0003          0.0013
         Turkey           97    -0.0006         -0.0006      0.0000          0.0004
3788742  Denmark         221     0.0001          0.0025      0.0003          0.0008
         Finland          70    -0.0010          0.0138      0.0000          0.0007
3788743  Belgium         143    -0.0010         -0.0002      0.0000          0.0005
         Russia           67     0.0005          0.0013     -0.0001          0.0013
3788744  Switzerland     158    -0.0004          0.0014      0.0000          0.0007
         Wales            72    -0.0030         -0.0015      0.0000          0.0008
3788745  Croatia         145    -0.0006          0.0007      0.0001          0.0003
         England         10

**Result of Checks 1 and 2**

Check 1 confirms that the two coordinate systems do not line up the same way for both teams. For one team in every match (the home team), SPADL keeps x as it is and flips y. For the other team (the away team), SPADL mirrors x end to end and keeps y as it is. This happens because socceraction flips every away-team action so that the away team attacks toward x = 0, while freeze frames are always drawn with the acting team attacking toward x = 120.

This matters because the candidate VAEP values were computed by placing each teammate's freeze-frame position, in StatsBomb yards, directly into the end location of a SPADL pass. For away-team passes, the hypothetical pass was therefore valued as if it went to the mirror-image spot on the pitch. For home-team passes, the direction was right, but the y axis was upside down and the scale was in yards rather than metres.

Check 2 shows how this plays out. For the teammate who actually received the ball, the typical gap between the hypothetical value and the real value is close to zero for every team, since most passes have a VAEP value near zero wherever they end. However, the average hypothetical value is clearly inflated for several away teams, for example Italy (0.0085 against a real average of -0.0010) and Finland (0.0138 against -0.0010). So most candidate values are barely affected, but a minority are badly wrong.

Because RQ2's forward-progressive test compares candidate_value with real_value, and both are usually close to zero, even small errors could flip that comparison. Check 3 measures how much.

Check 3: rebuilding the candidate values in the correct coordinates

The VAEP models were trained in setup_euro2024.ipynb but never saved. They are rebuilt here with the same training files, file order, train/test split and settings, then saved so this only happens once.

To confirm the rebuild is exact, two things are recomputed for five matches and compared with the stored numbers. The first is the old candidate value, using the same unconverted coordinates. The second is the real pass value. Both should match to within rounding. Then each candidate value is recomputed after converting the freeze-frame position into SPADL coordinates for that team's orientation.

The values are computed in batches rather than one window at a time. This gives exactly the same numbers as the original loop but runs far faster, which matters if all 166 matches need recomputing.

In [11]:
import os
import pickle
import socceraction.vaep.features as fs
from socceraction.vaep.formula import value
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

feature_functions = [
    fs.actiontype_onehot,
    fs.result_onehot,
    fs.startlocation,
    fs.endlocation,
    fs.team,
    fs.time_delta,
]

VAEP_MODEL_PATH = '../data/results/vaep_models_ext.pkl'

if os.path.exists(VAEP_MODEL_PATH):
    with open(VAEP_MODEL_PATH, 'rb') as f:
        models_ext = pickle.load(f)
    print(f"Loaded saved VAEP models from {VAEP_MODEL_PATH}")
else:
    # exactly as in setup_euro2024.ipynb: same file order, same split, same settings
    match_files_ext = glob.glob('../data/matches/match_*.pkl') + glob.glob('../data/matches_extended/match_*.pkl')
    print(f"{len(match_files_ext)} match files (should be 166)")
    X_parts, Y_parts = [], []
    for f in match_files_ext:
        d = pd.read_pickle(f)
        X_parts.append(d['X'])
        Y_parts.append(d['Y'])
    X_full_ext = pd.concat(X_parts, ignore_index=True)
    Y_full_ext = pd.concat(Y_parts, ignore_index=True)

    X_train_ext, X_test_ext, Y_train_ext, Y_test_ext = train_test_split(
        X_full_ext, Y_full_ext, test_size=0.2, random_state=42)

    t0 = time.time()
    models_ext = {
        'scores': GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
                  .fit(X_train_ext, Y_train_ext['scores']),
        'concedes': GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42)
                    .fit(X_train_ext, Y_train_ext['concedes']),
    }
    print(f"Trained in {time.time() - t0:.0f}s")
    for label in ['scores', 'concedes']:
        auc = roc_auc_score(Y_test_ext[label], models_ext[label].predict_proba(X_test_ext)[:, 1])
        print(f"{label}: test AUC = {auc:.3f}")

    with open(VAEP_MODEL_PATH, 'wb') as f:
        pickle.dump(models_ext, f)
    print(f"Saved to {VAEP_MODEL_PATH}")

Loaded saved VAEP models from ../data/results/vaep_models_ext.pkl


In [12]:
def batch_window_values(m_actions, pass_idx, end_x, end_y, models):
    """VAEP value of the last action in each 3-action window m_actions.loc[i-2:i],
    with that action's end moved to (end_x, end_y).

    Gives exactly the same number as the loop in setup_euro2024.ipynb (one window
    at a time through fs.gamestates, the feature functions and value()), including
    the padding socceraction applies at the start of a window or a period, but
    values thousands of windows in one call.
    """
    pass_idx = np.asarray(pass_idx)
    n = len(pass_idx)
    r0 = m_actions.loc[pass_idx - 2].reset_index(drop=True)
    r1 = m_actions.loc[pass_idx - 1].reset_index(drop=True)
    r2 = m_actions.loc[pass_idx].reset_index(drop=True)
    r2['end_x'] = np.asarray(end_x, dtype=float)
    r2['end_y'] = np.asarray(end_y, dtype=float)
    rows = pd.concat([r0, r1, r2], ignore_index=True)   # row number = role * n + window

    w = np.arange(n)
    i0, i1, i2 = w, n + w, 2 * n + w
    same01 = r0['period_id'].values == r1['period_id'].values
    same12 = r1['period_id'].values == r2['period_id'].values

    # the middle and last action of each window, with their two (padded) previous actions
    cur = np.column_stack([i1, i2]).ravel()
    prev1 = np.column_stack([np.where(same01, i0, i1),
                             np.where(same12, i1, i2)]).ravel()
    prev2 = np.column_stack([np.where(same01, i0, i1),
                             np.where(same12, np.where(same01, i0, i1), i2)]).ravel()

    gs = [rows.iloc[ix].reset_index(drop=True) for ix in (cur, prev1, prev2)]
    X = pd.concat([f(gs) for f in feature_functions], axis=1)
    ps = models['scores'].predict_proba(X)[:, 1]
    pc = models['concedes'].predict_proba(X)[:, 1]
    v = value(gs[0], pd.Series(ps), pd.Series(pc))['vaep_value'].values
    return v[1::2]   # keep the last action of each window


def sb_to_spadl(x, y, mirrored):
    """StatsBomb yards (acting team attacking toward x=120) to SPADL metres.
    Same formula socceraction uses, followed by its direction-of-play flip,
    which it applies to every away-team action."""
    x = np.clip((np.asarray(x, dtype=float) - 0.5) / 120 * 105, 0, 105)
    y = np.clip(68 - (np.asarray(y, dtype=float) - 0.5) / 80 * 68, 0, 68)
    mirrored = np.asarray(mirrored, dtype=bool)
    return np.where(mirrored, 105 - x, x), np.where(mirrored, 68 - y, y)


def mirrored_teams(m_actions, ev):
    """team_ids whose SPADL actions are mirrored relative to StatsBomb (the away team),
    found the same way as Check 1: whichever conversion lands closer to the SPADL start."""
    p = m_actions[m_actions['type_name'] == 'pass'].merge(
        ev[['id', 'location']], left_on='original_event_id', right_on='id')
    out = set()
    for team_id, g in p.groupby('team_id'):
        hx, _ = sb_to_spadl(g['location'].str[0], g['location'].str[1], False)
        ax_, _ = sb_to_spadl(g['location'].str[0], g['location'].str[1], True)
        if np.median(np.abs(g['start_x'] - ax_)) < np.median(np.abs(g['start_x'] - hx)):
            out.add(team_id)
    return out

In [13]:
recomputed = []
for mid in check_ids:
    m_actions = load_match_actions(mid).sort_values('action_id').reset_index(drop=True)
    ev = cached_events(mid)
    away = mirrored_teams(m_actions, ev)

    pass_rows = m_actions[m_actions['type_name'] == 'pass']
    idx_of = pd.Series(pass_rows.index, index=pass_rows['original_event_id'])
    idx_of = idx_of[~idx_of.index.duplicated()]

    c = candidates_full[candidates_full['match_id'] == mid].copy()
    c['pass_idx'] = c['original_event_id'].map(idx_of)
    c = c.dropna(subset=['pass_idx'])
    c['pass_idx'] = c['pass_idx'].astype(int)
    c['mirrored'] = m_actions.loc[c['pass_idx'], 'team_id'].isin(away).values

    # 1. the old calculation: freeze-frame yards dropped straight into SPADL (should match the stored values)
    c['old_value_rebuilt'] = batch_window_values(
        m_actions, c['pass_idx'], c['cand_x'], c['cand_y'], models_ext)

    # 2. the real pass, unchanged (should match the stored real_value)
    c['real_value_rebuilt'] = batch_window_values(
        m_actions, c['pass_idx'],
        m_actions.loc[c['pass_idx'], 'end_x'], m_actions.loc[c['pass_idx'], 'end_y'], models_ext)

    # 3. the corrected calculation: freeze-frame position converted to SPADL first
    sx, sy = sb_to_spadl(c['cand_x'], c['cand_y'], c['mirrored'])
    c['candidate_value_fixed'] = batch_window_values(m_actions, c['pass_idx'], sx, sy, models_ext)

    recomputed.append(c)

check3 = pd.concat(recomputed)
print(f"{len(check3)} of {(candidates_full['match_id'].isin(check_ids)).sum()} candidate rows matched to a SPADL pass")
print()
print("Validation (differences should be about 0, e.g. below 1e-9):")
print(f"  old candidate value, max |rebuilt - stored|: {(check3['old_value_rebuilt'] - check3['candidate_value']).abs().max():.2e}")
rv = check3.dropna(subset=['real_value'])
print(f"  real value,          max |rebuilt - stored|: {(rv['real_value_rebuilt'] - rv['real_value']).abs().max():.2e}")

17751 of 17751 candidate rows matched to a SPADL pass

Validation (differences should be about 0, e.g. below 1e-9):
  old candidate value, max |rebuilt - stored|: 0.00e+00
  real value,          max |rebuilt - stored|: 0.00e+00


In [14]:
XP_THRESHOLD = 0.05
c = check3.dropna(subset=['real_value']).copy()
c['side'] = np.where(c['mirrored'], 'away (mirrored)', 'home')

xt_up = c['xt_target'] > c['xt_origin']
gate = c['visible'] & ~c['offside'].fillna(True) & (c['xp_value'] >= XP_THRESHOLD)
c['fp_old'] = xt_up & (c['candidate_value'] > c['real_value'])
c['fp_new'] = xt_up & (c['candidate_value_fixed'] > c['real_value'])
c['legit_old'] = gate & c['fp_old']
c['legit_new'] = gate & c['fp_new']

print("1. How much the candidate values moved")
print(c.groupby('side').apply(lambda g: pd.Series({
    'rows': len(g),
    'mean_old': g['candidate_value'].mean(),
    'mean_fixed': g['candidate_value_fixed'].mean(),
    'mean_real': g['real_value'].mean(),
    'corr_old_fixed': np.corrcoef(g['candidate_value'], g['candidate_value_fixed'])[0, 1],
})).round(4).to_string())

print("\n2. Receiver check after the fix (candidate nearest the pass end, within 5 yards)")
ends = pd.concat([cached_events(m)[['id', 'pass_end_location']] for m in check_ids])
r = c.merge(ends, left_on='original_event_id', right_on='id')
r['d_end'] = np.hypot(r['cand_x'] - r['pass_end_location'].str[0], r['cand_y'] - r['pass_end_location'].str[1])
r = r.loc[r.groupby('original_event_id')['d_end'].idxmin()]
r = r[r['d_end'] < 5]
print(r.groupby('side').apply(lambda g: pd.Series({
    'passes': len(g),
    'mean_gap_old': (g['candidate_value'] - g['real_value']).mean(),
    'mean_gap_fixed': (g['candidate_value_fixed'] - g['real_value']).mean(),
    'median_abs_gap_old': (g['candidate_value'] - g['real_value']).abs().median(),
    'median_abs_gap_fixed': (g['candidate_value_fixed'] - g['real_value']).abs().median(),
})).round(4).to_string())

print("\n3. Candidate-level flags that change")
for col in ['fp', 'legit']:
    changed = (c[f'{col}_old'] != c[f'{col}_new'])
    print(f"  {col:5s}: old {c[f'{col}_old'].mean():.1%}  fixed {c[f'{col}_new'].mean():.1%}  "
          f"rows that changed {changed.mean():.1%}")

print("\n4. Pass-level RQ2 rate (share of passes with a legitimate forward-progressive option)")
pl = c.groupby(['match_id', 'original_event_id', 'side']).agg(
    old=('legit_old', 'any'), fixed=('legit_new', 'any')).reset_index()
print(pl.groupby('side')[['old', 'fixed']].mean().round(3).to_string())
print(f"  all passes: old {pl['old'].mean():.1%}  fixed {pl['fixed'].mean():.1%}  "
      f"passes whose answer changed {(pl['old'] != pl['fixed']).mean():.1%}")

1. How much the candidate values moved
                   rows  mean_old  mean_fixed  mean_real  corr_old_fixed
side                                                                    
away (mirrored)  6713.0    0.0036     -0.0001    -0.0002          0.4255
home             8088.0    0.0024     -0.0000    -0.0006          0.9429

2. Receiver check after the fix (candidate nearest the pass end, within 5 yards)
                 passes  mean_gap_old  mean_gap_fixed  median_abs_gap_old  median_abs_gap_fixed
side                                                                                           
away (mirrored)   552.0        0.0042          0.0001              0.0009                   0.0
home              552.0        0.0013          0.0001              0.0006                   0.0

3. Candidate-level flags that change
  fp   : old 44.8%  fixed 37.6%  rows that changed 12.9%
  legit: old 15.9%  fixed 12.7%  rows that changed 5.3%

4. Pass-level RQ2 rate (share of passes with a legi

**Result of Check 3**

The rebuild is exact. Recomputing the old candidate values and the real pass values with the rebuilt VAEP models reproduces the stored numbers with a maximum difference of zero, so the models, the batched calculation and the matching of candidates to SPADL passes are all identical to the original pipeline.

The coordinate fix works. After converting each freeze-frame position into SPADL coordinates, the hypothetical pass to the actual receiver is worth almost exactly what the real pass was worth: the mean gap falls from 0.0042 to 0.0001 for away teams and from 0.0013 to 0.0001 for home teams, and the median absolute gap falls to zero. That is what should happen, since the two are nearly the same action.

Both teams were affected, in different ways. Away-team candidates were valued at the mirror-image point on the pitch, so their old values are only weakly related to the corrected ones (correlation 0.43). Home-team candidates had the right direction but were placed about 14 percent further up the pitch, because yards were read as metres, which made forward positions look more valuable than they were (correlation 0.94). In both cases the old average candidate value sat above the real pass value, and after the fix it matches it.

The effect on RQ2 is large enough to matter. On these five matches, 12.9 percent of candidates change their forward-progressive status, the share of legitimate forward options falls from 15.9 to 12.7 percent, and the pass-level rate falls from 51.1 to 43.7 percent, with 14.2 percent of passes changing their answer. The old values overstated forward options, so the corrected forward-bias rate will be lower than 45.5 percent.

The candidate values for all 166 matches are therefore recomputed in layer1_visual_constraint.ipynb, and RQ1 and RQ2 are rerun from the corrected values. Real pass values, offside, xP, xT, the vision cone, the LSTM and the receiver matching used for blind passes all work in StatsBomb coordinates and are not affected.

Checks 2 and 3 read candidates_full_old_coords.pkl, the copy of the candidate table saved before the fix, so they still compare against the original values after layer 2 is rerun.

**Outcome of the full recompute**

The corrected values were computed for all 166 matches at the end of layer1_visual_constraint.ipynb, and layer 2 and the three-way split were rerun from them.

- The recompute reproduced every stored old value exactly across all 664510 candidate rows, and every match had exactly one mirrored (away) team.
- RQ1 is essentially unchanged: the vision cone changes the best alternative in 57.9 percent of passes (58.3 percent before), with d = 0.282 (0.281 before). The values themselves are smaller, since the old ones were inflated.
- The RQ2 forward-bias rate falls from 45.5 to 38.5 percent (95 percent CI 38.2 to 38.8), with the same ordering across the three tournaments.
- In the three-way split, visible candidates no longer stand out: 33.6 percent of visible, 36.2 percent of memory-feasible and 34.8 percent of memory-infeasible candidates are legitimate forward options, and the differences follow how often each group's players are in forward positions.

The blind-pass test below uses StatsBomb coordinates throughout (pass end locations and freeze-frame positions), so it does not depend on the VAEP values.

#### The blind-pass test

A blind pass is a completed pass to a teammate who was outside the passer's 120 degree vision cone at the moment of the pass. If passers use short-term memory of where their teammates are, then when they pass blind, they should tend to pick teammates whose position could have been remembered. So the receivers of blind passes should be memory-feasible more often than a teammate picked at random from the invisible teammates who were available on the same pass.

The receiver is identified from the freeze frame: the teammate standing nearest to where the pass ended, within 5 yards. Both positions are in StatsBomb coordinates, so this does not depend on the VAEP values. A receiver is flagged as ambiguous if a second teammate was almost as close.

Memory feasibility comes from the same cross-fitted LSTM predictions used in the three-way split: a teammate is memory-feasible if they were seen within the previous two seconds and the LSTM places them within τ of their true position.

In [15]:
RECEIVER_RADIUS = 5.0   # yards between the pass end location and the receiver's freeze-frame position
AMBIGUOUS_GAP = 2.0     # yards: if the second-nearest teammate is this close to the nearest, the receiver is ambiguous

cands = pd.read_pickle('../data/results/candidates_full.pkl')
cands['match_id'] = cands['match_id'].astype(str)
keys = ['match_id', 'original_event_id', 'cand_x', 'cand_y']
cands = cands.drop_duplicates(keys)

# cross-fitted LSTM predictions for every invisible teammate (visible teammates get NaN)
cf = pd.read_pickle('../data/results/rq3_crossfit_predictions.pkl')
cf['match_id'] = cf['match_id'].astype(str)
cands = cands.merge(cf[keys + ['n_hist', 'horizon_s', 'err_last_m', 'err_lstm_m']].drop_duplicates(keys),
                    on=keys, how='left')

# where each pass actually ended, from the StatsBomb events (same coordinates as the freeze frames)
ends = []
for mid in cands['match_id'].unique():
    ev = cached_events(mid)
    ev = ev[ev['type'] == 'Pass'][['id', 'pass_end_location']]
    ends.append(ev.rename(columns={'id': 'original_event_id'}))
ends = pd.concat(ends, ignore_index=True)
cands = cands.merge(ends, on='original_event_id', how='left')
cands['d_end'] = np.hypot(cands['cand_x'] - cands['pass_end_location'].str[0],
                          cands['cand_y'] - cands['pass_end_location'].str[1])

# per pass: the nearest teammate to the pass end, and how far the second-nearest was
cands = cands.sort_values(['match_id', 'original_event_id', 'd_end'])
cands['rank_end'] = cands.groupby(['match_id', 'original_event_id']).cumcount()
second = cands[cands['rank_end'] == 1].set_index(['match_id', 'original_event_id'])['d_end']
passes = cands[cands['rank_end'] == 0].set_index(['match_id', 'original_event_id'])
passes = passes.assign(d_second=second.reindex(passes.index))
passes['receiver_found'] = passes['d_end'] < RECEIVER_RADIUS
passes['ambiguous'] = passes['receiver_found'] & ((passes['d_second'] - passes['d_end']) < AMBIGUOUS_GAP)

cands['is_receiver'] = False
rec_idx = cands.index[(cands['rank_end'] == 0) & (cands['d_end'] < RECEIVER_RADIUS)]
cands.loc[rec_idx, 'is_receiver'] = True

found = passes[passes['receiver_found']]
print(f"{len(passes)} passes")
print(f"receiver identified (a teammate within {RECEIVER_RADIUS:.0f} yards of the pass end): "
      f"{len(found)} ({len(found) / len(passes):.1%})")
print(f"  of those, ambiguous (another teammate within {AMBIGUOUS_GAP:.0f} yards of the same distance): "
      f"{found['ambiguous'].mean():.1%}")
print(f"  median distance from receiver to pass end: {found['d_end'].median():.2f} yards")
print(f"blind passes (receiver outside the vision cone): {(~found['visible']).sum()} "
      f"({(~found['visible']).mean():.1%} of passes with a receiver)")

98652 passes
receiver identified (a teammate within 5 yards of the pass end): 50610 (51.3%)
  of those, ambiguous (another teammate within 2 yards of the same distance): 3.2%
  median distance from receiver to pass end: 2.88 yards
blind passes (receiver outside the vision cone): 24142 (47.7% of passes with a receiver)


**Result: identifying the receiver**

A teammate stood within 5 yards of the pass end for 50610 of the 98652 passes (51.3 percent), with a median distance of 2.9 yards, and only 3.2 percent of these were ambiguous. The other half of passes have no teammate that close, mostly because the receiver ran onto the ball and was further away at the moment of the pass. Those passes are left out of the test rather than assigned a guessed receiver.

Of the passes with a receiver, 24142 (47.7 percent) went to a teammate outside the passer's vision cone. These are the blind passes used in the test below.

In [16]:
TAU_MAIN = 4.0
inv = cands[~cands['visible']].copy()
inv['has_history'] = inv['n_hist'] >= 1


def blind_pass_test(inv, flag, restrict=None, n_boot=5000, seed=0):
    """For blind passes, is the receiver `flag` more often than a teammate picked at random
    from the invisible teammates available at that moment?
    restrict: optional column; only teammates where it is True are counted (e.g. has_history)."""
    d = inv if restrict is None else inv[inv[restrict]]
    g = d.groupby(['match_id', 'original_event_id'])
    per_pass = pd.DataFrame({
        'n_options': g.size(),
        'has_receiver': g['is_receiver'].any(),
        'expected': g[flag].mean(),                                   # chance a random pick has the flag
        'observed': g.apply(lambda x: x.loc[x['is_receiver'], flag].any()),
    }).reset_index()
    per_pass = per_pass[per_pass['has_receiver'] & (per_pass['n_options'] >= 2)]
    per_pass['diff'] = per_pass['observed'].astype(float) - per_pass['expected']

    per_match = per_pass.groupby('match_id')['diff'].agg(['sum', 'count'])
    ids = per_match.index.to_numpy()
    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot)
    for b in range(n_boot):
        sub = per_match.loc[rng.choice(ids, size=len(ids), replace=True)]
        boot[b] = sub['sum'].sum() / sub['count'].sum()
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return pd.Series({
        'blind_passes': len(per_pass),
        'mean_options': per_pass['n_options'].mean(),
        'observed_share': per_pass['observed'].mean(),
        'expected_share': per_pass['expected'].mean(),
        'difference': per_pass['diff'].mean(),
        'ci_low': lo,
        'ci_high': hi,
    })


rows = {}
for tau in [3.0, 4.0, 5.0]:
    inv[f'feasible_{tau:.0f}'] = inv['has_history'] & (inv['err_lstm_m'] < tau)
    rows[f'memory-feasible, tau = {tau:.0f} m'] = blind_pass_test(inv, f'feasible_{tau:.0f}')
rows['has an earlier point (seen in last 2 s)'] = blind_pass_test(inv, 'has_history')
rows['within 4 m, among teammates with an earlier point'] = blind_pass_test(inv, 'feasible_4', restrict='has_history')
print(pd.DataFrame(rows).T.round(3).to_string())

                                                   blind_passes  mean_options  observed_share  expected_share  difference  ci_low  ci_high
memory-feasible, tau = 3 m                              23509.0         5.170           0.337           0.359      -0.022  -0.025   -0.019
memory-feasible, tau = 4 m                              23509.0         5.170           0.421           0.444      -0.023  -0.026   -0.019
memory-feasible, tau = 5 m                              23509.0         5.170           0.478           0.498      -0.019  -0.022   -0.016
has an earlier point (seen in last 2 s)                 23509.0         5.170           0.586           0.588      -0.002  -0.004   -0.000
within 4 m, among teammates with an earlier point       13702.0         5.265           0.720           0.757      -0.037  -0.042   -0.031


**Result: do blind passes go to remembered teammates?**

On 23509 blind passes with at least two invisible teammates available (about 5 on average), the receiver was memory-feasible at τ = 4 m in 42.1 percent of cases, against 44.4 percent for a teammate picked at random from the same invisible teammates. The difference is -2.3 percentage points (95 percent interval -2.6 to -1.9, resampling whole matches), and it is the same at τ = 3 m (-2.2) and τ = 5 m (-1.9). So blind passes do not favour teammates whose position could be recovered from memory. If anything, they go slightly less often to them.

The two parts of memory feasibility behave differently. Receivers were seen within the previous two seconds exactly as often as the other invisible teammates (58.6 against 58.8 percent, difference -0.2). The whole gap comes from the second part: among teammates who had been seen, the LSTM places the receiver within 4 m less often (72.0 against 75.7 percent, difference -3.7).

A likely explanation is movement. A receiver is often moving to get on the ball, and a moving player is harder to predict from where they were last seen. If so, the negative gap reflects how memory feasibility is measured, since low prediction error favours players standing still, rather than passers avoiding teammates they remember. The next cells check this, and test the receiver identification itself.

**Checking the result**

Two checks. First, the same random-pick test is applied to visibility: across all passes with an identified receiver, is the receiver inside the vision cone more often than a teammate picked at random from the same freeze frame? This tests whether the vision cone captures anything about who players pass to. Second, for blind passes, the distance each invisible teammate had moved since they were last seen is compared between receivers and the other invisible teammates, to test whether receivers are harder to predict because they are moving.

Then the receiver is re-identified at radii of 3, 5, 8 and 10 yards, to check that none of the results depend on the 5 yard choice.

In [17]:
# 1. do passers pick visible teammates more often than chance? (all passes with a receiver)
cands['visible_flag'] = cands['visible'].astype(bool)
print("Receiver inside the vision cone, against a teammate picked at random on the same pass:")
print(blind_pass_test(cands, 'visible_flag').round(3).to_string())

# 2. how far had blind-pass receivers moved since they were last seen, compared with the other
#    invisible teammates on the same passes? (only teammates seen in the last 2 s)
on_blind = inv.groupby(['match_id', 'original_event_id'])['is_receiver'].transform('any')
seen = inv[on_blind & inv['has_history']]
print("\nInvisible teammates seen in the last 2 s, on blind passes:")
print(seen.groupby('is_receiver').agg(
    teammates=('err_lstm_m', 'size'),
    moved_since_seen_m=('err_last_m', 'mean'),
    lstm_error_m=('err_lstm_m', 'mean'),
    time_since_seen_s=('horizon_s', 'mean'),
).rename(index={False: 'other invisible teammates', True: 'receiver'}).round(2).to_string())

Receiver inside the vision cone, against a teammate picked at random on the same pass:
blind_passes      50588.000
mean_options          6.943
observed_share        0.523
expected_share        0.345
difference            0.178
ci_low                0.172
ci_high               0.185

Invisible teammates seen in the last 2 s, on blind passes:
                           teammates  moved_since_seen_m  lstm_error_m  time_since_seen_s
is_receiver                                                                              
other invisible teammates      60872                3.00          2.91                0.9
receiver                       14130                3.28          3.18                0.9


**Result of the checks**

The vision cone captures something real about who players pass to. Across 50588 passes with an identified receiver, the receiver was inside the passer's vision cone 52.3 percent of the time, against 34.5 percent for a teammate picked at random from the same freeze frame. The difference is 17.8 percentage points (95 percent interval 17.2 to 18.5, resampling whole matches), so players pass to teammates inside their cone about one and a half times as often as chance would give. One caveat is that the cone points along the passer's direction of movement, and play tends to flow in one direction, so part of this gap may reflect passing in the direction of play rather than visibility alone.

The movement explanation for the blind-pass result holds. Among invisible teammates on blind passes who had been seen in the previous two seconds, receivers had moved 3.28 m since they were last seen, against 3.00 m for the other invisible teammates, over the same time (0.9 s on average). Their LSTM error is higher by almost the same amount (3.18 against 2.91 m). So receivers are harder to place because they are moving more, typically getting into position to receive the ball, not because passers avoid teammates whose position they remember.

Taken together: passers clearly favour teammates they can see, but when they pass to a teammate they cannot see, there is no sign that they favour the ones whose position could be recovered from the previous two seconds. The slightly lower memory-feasibility of blind-pass receivers reflects the fact that receivers move more, which makes them harder to predict, rather than evidence against the use of memory. On this data, short-term memory as measured by position recoverability does not explain who receives blind passes.

In [18]:
def receivers_within(radius):
    """Copy of the candidate table with the receiver re-identified at a different radius."""
    c = cands.drop(columns=['is_receiver']).copy()
    c['is_receiver'] = (c['rank_end'] == 0) & (c['d_end'] < radius)
    return c


rows = {}
for radius in [3.0, 5.0, 8.0, 10.0]:
    c = receivers_within(radius)
    i = c[~c['visible']].copy()
    i['has_history'] = i['n_hist'] >= 1
    i['feasible_4'] = i['has_history'] & (i['err_lstm_m'] < 4.0)
    rows[(radius, 'receiver visible (all passes)')] = blind_pass_test(c, 'visible_flag', n_boot=2000)
    rows[(radius, 'memory-feasible, tau = 4 m')] = blind_pass_test(i, 'feasible_4', n_boot=2000)
    rows[(radius, 'within 4 m, among seen recently')] = blind_pass_test(i, 'feasible_4', restrict='has_history', n_boot=2000)
    print(f"radius {radius:.0f} yards done")

sens = pd.DataFrame(rows).T
sens.index.names = ['radius_yd', 'test']
print(sens[['blind_passes', 'observed_share', 'expected_share', 'difference', 'ci_low', 'ci_high']].round(3).to_string())

radius 3 yards done
radius 5 yards done
radius 8 yards done
radius 10 yards done
                                           blind_passes  observed_share  expected_share  difference  ci_low  ci_high
radius_yd test                                                                                                      
3.0       receiver visible (all passes)         26935.0           0.522           0.345       0.176   0.169    0.183
          memory-feasible, tau = 4 m            12553.0           0.439           0.459      -0.020  -0.024   -0.016
          within 4 m, among seen recently        7496.0           0.732           0.765      -0.032  -0.038   -0.026
5.0       receiver visible (all passes)         50588.0           0.523           0.345       0.178   0.172    0.185
          memory-feasible, tau = 4 m            23509.0           0.421           0.444      -0.023  -0.026   -0.019
          within 4 m, among seen recently       13702.0           0.720           0.757      -0.037 

**Result: sensitivity to the receiver radius**

All three results hold whether the receiver is identified within 3, 5, 8 or 10 yards of the pass end, even though the number of passes with a receiver nearly triples from 26935 to 78281.

The receiver is inside the vision cone 17.6 to 18.7 percentage points more often than a random teammate at every radius. On blind passes, the receiver is memory-feasible 2.0 to 2.8 points less often than a random invisible teammate, and among teammates seen in the previous two seconds, is placed within 4 m 3.2 to 4.4 points less often. Every interval excludes zero.

The memory gap widens slightly as the radius grows. That is what the movement explanation predicts: a wider radius takes in more receivers who were further from the pass end at the moment of the pass because they ran onto the ball, and these are the players who are hardest to predict from where they were last seen. So the negative gap is not an artifact of the 5 yard choice. It is a consistent feature of who receives passes.

## Conclusion for RQ3, part 2

Three findings, all robust to the receiver radius and to the choice of τ.

1. Passers strongly favour teammates they can see. The receiver is inside the 120 degree vision cone 52.3 percent of the time, against 34.5 percent for a teammate picked at random from the same freeze frame, a difference of 17.8 percentage points. This supports the vision cone as a meaningful model of which options a passer considers, with the caveat that it is built from movement direction and so partly reflects passing in the direction of play.

2. When passers pass to a teammate outside the cone, they do not favour teammates whose position could be recovered from the previous two seconds. Blind-pass receivers were seen recently exactly as often as the other invisible teammates (58.6 against 58.8 percent), and are memory-feasible slightly less often (42.1 against 44.4 percent at τ = 4 m).

3. That small deficit is explained by movement, not avoidance. Receivers had moved further since they were last seen than the other invisible teammates over the same time (3.28 against 3.00 m in 0.9 s), which makes them harder for the LSTM to place.

Together with part 1 (the LSTM places invisible teammates about 0.1 m better than assuming they stood still) and the three-way split (memory feasibility does not separate teammates by how often they were missed forward options), the RQ3 answer is that short-term positional memory, as far as this data can measure it, adds little to the vision cone. Positions over the last two seconds are recoverable to about 3 m, but whether a teammate's position was recoverable does not predict who receives a blind pass or who is left unused. Freeze frames only cover a second or two of history per player, so this is a statement about very short-term memory. Longer-range tactical knowledge of where teammates are likely to be, which may be what blind passes actually rely on, is outside what this data can test.